In [ ]:
import os
		
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForSeq2Seq, TrainingArguments, Trainer

In [ ]:
ds = Dataset.load_from_disk("../data/alpaca_data_zh/")
ds

In [ ]:
ds = ds.train_test_split(test_size=0.2)
ds['train'][:3]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(r"E:\modelscope\models\ZhipuAI\chatglm3-6b", trust_remote_code=True)
tokenizer

参考修改：https://www.doubao.com/chat/33510768991908098

In [ ]:
tokenizer.eos_token_id, tokenizer(tokenizer.eos_token)

In [ ]:
def process_func(example):
    MAX_LENGTH = 256
    input_ids, attention_mask, labels = [], [], []
    instruction = "\n".join([example["instruction"], example["input"]]).strip()     # query
    instruction = tokenizer.build_chat_input(instruction, history=[], role="user")  # [gMASK]sop<|user|> \n query<|assistant|>
    response = tokenizer("\n" + example["output"], add_special_tokens=False)        # \n response, 缺少eos token
    input_ids = instruction["input_ids"][0].numpy().tolist() + response["input_ids"] + [tokenizer.eos_token_id]
    attention_mask = instruction["attention_mask"][0].numpy().tolist() + response["attention_mask"] + [1]
    labels = [-100] * len(instruction["input_ids"][0].numpy().tolist()) + response["input_ids"] + [tokenizer.eos_token_id]
    if len(input_ids) > MAX_LENGTH:
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [ ]:
tokenized_ds = ds.map(process_func, remove_columns=ds['train'].column_names)
tokenized_ds

In [ ]:
tokenizer.decode(tokenized_ds['train'][1]["input_ids"])

In [ ]:
tokenizer.decode(list(filter(lambda x: x != -100, tokenized_ds['train'][1]["labels"])))

In [ ]:
import torch

"""
新版本中需要将modeling_chatglm源码中的613行部分进行调整，代码如下：

```
if not kv_caches:
    kv_caches = [None for _ in range(self.num_layers)]
else:
    kv_caches = kv_caches[1]
```

如果不进行调整，后续chat阶段会报错
"""
# 多卡情况，可以去掉device_map="auto"，否则会将模型拆开
model = AutoModelForCausalLM.from_pretrained(r"E:\modelscope\models\ZhipuAI\chatglm3-6b", trust_remote_code=True, torch_dtype=torch.bfloat16)

In [ ]:
for name, param in model.named_parameters():
    print(name)

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model, PeftModel

config = LoraConfig(target_modules=["query_key_value"], modules_to_save=["post_attention_layernorm"])
config

In [ ]:
model = get_peft_model(model, config)

In [ ]:
for name, parameter in model.named_parameters():
    print(name)

In [ ]:
model.print_trainable_parameters()

In [ ]:
args = TrainingArguments(
    output_dir="./chatbot",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=16,
    logging_steps=50,
    eval_strategy='steps',
    num_train_epochs=1,
    learning_rate=1e-4,
    remove_unused_columns=False,
    save_strategy="epoch"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_ds['train'].select(range(5000)),
    eval_dataset=tokenized_ds['test'].select(range(5000)),
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
)

In [ ]:
trainer.train()

In [ ]:
model.eval()
print(model.chat(tokenizer, "数学考试怎么考高分？", history=[])[0])